### SCD type 1
- The previous records are discarded and only the new records are added to the target table
- i.e if the value of a column changes in a record then the new value is added to the record when match is found using the merge statement 

In [0]:
%sql
create or replace table datamodeling.default.scd_source
(
    prod_id int,
    prod_name string,
    prod_cat string,
    processDateTime timestamp
)

In [0]:
%sql
insert into datamodeling.default.scd_source
values
(1, 'prod1', 'cat1', current_timestamp()),
(2, 'prod2', 'cat2', current_timestamp()),
(3, 'prod3', 'cat3', current_timestamp())

num_affected_rows,num_inserted_rows
3,3


In [0]:
%sql
select * from datamodeling.default.scd_source

prod_id,prod_name,prod_cat,processDateTime
1,prod1,cat1,2026-05-11T09:40:30.005Z
2,prod2,cat2,2026-05-11T09:40:30.005Z
3,prod3,cat3,2026-05-11T09:40:30.005Z


#### Creating target table

In [0]:
%sql
create or replace table datamodeling.gold.scdtype1_table
(
    prod_id int,
    prod_name string,
    prod_cat string,
    processDateTime timestamp
)

#### Using merge command

In [0]:
%sql
merge into datamodeling.gold.scdtype1_table as dest
using datamodeling.default.scd_source as src
on dest.prod_id = src.prod_id
when matched and (
    src.processDateTime >= dest.processDateTime
) and(
    src.prod_id <> dest.prod_id
    or src.prod_name <> dest.prod_name
    or src.prod_cat <> dest.prod_cat
)
then update set *
when not matched then insert *

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
1,1,0,0


In [0]:
%sql
select * from datamodeling.gold.scdtype1_table

prod_id,prod_name,prod_cat,processDateTime
1,prod1,cat1,2026-05-11T09:40:30.005Z
2,prod2,cat2,2026-05-11T09:40:30.005Z
3,prod3,cat3,2026-05-11T09:40:30.005Z


##### Updating the value of a column

In [0]:
%sql
update datamodeling.default.scd_source
set prod_cat = 'newCategory'
where prod_id = 3

num_affected_rows
1


In [0]:
%sql
select * from datamodeling.default.scd_source

prod_id,prod_name,prod_cat,processDateTime
3,prod3,newCategory,2026-05-11T09:40:30.005Z
1,prod1,cat1,2026-05-11T09:40:30.005Z
2,prod2,cat2,2026-05-11T09:40:30.005Z


- Now, after using the merge command, the output will be

In [0]:
%sql
select * from datamodeling.gold.scdtype1_table

prod_id,prod_name,prod_cat,processDateTime
1,prod1,cat1,2026-05-11T09:40:30.005Z
2,prod2,cat2,2026-05-11T09:40:30.005Z
3,prod3,newCategory,2026-05-11T09:40:30.005Z


### SCD Type-2
- It contains start_date and end_date.
- When new value is updated in the table then the pevious record is marked as expired
- The information of the previous record is retained in type 2 unlike the type 1
- When the record is updated then the previous record is marked as expired with end_date and the new record starts with the start_date as curent_date.

In [0]:
%sql
create or replace table datamodeling.default.scdtype2_source(
    prod_id int,
    prod_name string,
    prod_cat string,
    processDate timestamp
)

In [0]:
%sql
insert into datamodeling.default.scdtype2_source values
(1,'prod1','cat1',current_timestamp()),
(2,'prod2','cat2',current_timestamp()),
(3,'prod3','cat3',current_timestamp())
    

num_affected_rows,num_inserted_rows
3,3


##### Creating target table with start_date, end_date and is_current

In [0]:
%sql
create or replace table datamodeling.gold.scdtype2_table(
    prod_id int,
    prod_name string,
    prod_cat string,
    processDate timestamp,
    start_date timestamp,
    end_date timestamp,
    is_current string
)

In [0]:
spark.sql("""
        select *,
                current_timestamp() as start_date,
                cast("3000-01-01" as timestamp) as end_date,
                'Y' as is_current
        from datamodeling.default.scdtype2_source""").createOrReplaceTempView("type2_src")


In [0]:
%sql
select * from type2_src

prod_id,prod_name,prod_cat,processDate,start_date,end_date,is_current
1,prod1,cat1,2026-05-11T09:42:12.464Z,2026-05-11T09:42:57.037Z,3000-01-01T00:00:00.000Z,Y
2,prod2,cat2,2026-05-11T09:42:12.464Z,2026-05-11T09:42:57.037Z,3000-01-01T00:00:00.000Z,Y
3,prod3,cat3,2026-05-11T09:42:12.464Z,2026-05-11T09:42:57.037Z,3000-01-01T00:00:00.000Z,Y


- 1. First mark the old records as N, and make the end_date as current date using the merge statement
- 2. Then insert the new records using another merge statement with start_date same as end_date of old record
- The behaviour of merge in delta lake is: first it identifies the matched an non-matched records, then only it applies updation process. So for scd type2, two merge are needed. First for marking the old records as N and second for updating the new records

In [0]:
%sql
merge into datamodeling.gold.scdtype2_table as dest
using type2_src as src
on dest.prod_id = src.prod_id and
dest.is_current = "Y"
when matched and (
    src.prod_name <> dest.prod_name or
    src.prod_cat <> dest.prod_cat or
    src.processDate<> dest.processDate
)
then update set
    dest.end_date = current_timestamp(),
    dest.is_current = "N"

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
1,1,0,0


In [0]:
%sql
merge into datamodeling.gold.scdtype2_table as dest
using type2_src as src
on dest.prod_id = src.prod_id and
dest.is_current = "Y"
when not matched then insert(
    prod_id,
    prod_name,
    prod_cat,
    processDate,
    start_date,
    end_date,
    is_current
)
values (
    src.prod_id,
    src.prod_name,
    src.prod_cat,
    src.processDate,
    src.start_date,
    src.end_date,
    src.is_current
)

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
1,0,0,1


In [0]:
%sql
select * from datamodeling.gold.scdtype2_table

prod_id,prod_name,prod_cat,processDate,start_date,end_date,is_current
1,prod1,cat1,2026-05-11T09:42:12.464Z,2026-05-11T09:43:20.307Z,3000-01-01T00:00:00.000Z,Y
2,prod2,cat2,2026-05-11T09:42:12.464Z,2026-05-11T09:43:20.307Z,3000-01-01T00:00:00.000Z,Y
3,prod3,cat3,2026-05-11T09:42:12.464Z,2026-05-11T09:43:20.307Z,3000-01-01T00:00:00.000Z,Y


In [0]:
%sql
update datamodeling.default.scdtype2_source
set prod_cat = 'newCategory'
where prod_id = 2

num_affected_rows
1


In [0]:
%sql
select * from type2_src


prod_id,prod_name,prod_cat,processDate,start_date,end_date,is_current
2,prod2,newCategory,2026-05-11T09:42:12.464Z,2026-05-11T09:45:35.661Z,3000-01-01T00:00:00.000Z,Y
1,prod1,cat1,2026-05-11T09:42:12.464Z,2026-05-11T09:45:35.661Z,3000-01-01T00:00:00.000Z,Y
3,prod3,cat3,2026-05-11T09:42:12.464Z,2026-05-11T09:45:35.661Z,3000-01-01T00:00:00.000Z,Y


- Now use the merge statements after changing the column value of the source

- Output after the first merge
- i.e marking the old record as N and updating the end_date column

In [0]:
%sql
select * from datamodeling.gold.scdtype2_table

prod_id,prod_name,prod_cat,processDate,start_date,end_date,is_current
1,prod1,cat1,2026-05-11T09:42:12.464Z,2026-05-11T09:43:20.307Z,3000-01-01T00:00:00.000Z,Y
3,prod3,cat3,2026-05-11T09:42:12.464Z,2026-05-11T09:43:20.307Z,3000-01-01T00:00:00.000Z,Y
2,prod2,cat2,2026-05-11T09:42:12.464Z,2026-05-11T09:43:20.307Z,2026-05-11T09:45:56.090Z,N


- Output after the second merge i.e inserting the new records

In [0]:
%sql
select * from datamodeling.gold.scdtype2_table

prod_id,prod_name,prod_cat,processDate,start_date,end_date,is_current
1,prod1,cat1,2026-05-11T09:42:12.464Z,2026-05-11T09:43:20.307Z,3000-01-01T00:00:00.000Z,Y
3,prod3,cat3,2026-05-11T09:42:12.464Z,2026-05-11T09:43:20.307Z,3000-01-01T00:00:00.000Z,Y
2,prod2,cat2,2026-05-11T09:42:12.464Z,2026-05-11T09:43:20.307Z,2026-05-11T09:45:56.090Z,N
2,prod2,newCategory,2026-05-11T09:42:12.464Z,2026-05-11T09:46:43.999Z,3000-01-01T00:00:00.000Z,Y
